# Quickstart：用真实 EEG 观察睁闭眼 Alpha

这个 Notebook 从 PhysioNet EEGMMIDB 下载一名受试者的睁眼与闭眼基线。首次运行需要联网。数据会进入 MNE 缓存，不会写入 Git。

In [ ]:
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci

mne.set_log_level('WARNING')
paths = eegbci.load_data(subjects=1, runs=[1, 2], update_path=True)
eyes_open = mne.io.read_raw_edf(paths[0], preload=True)
eyes_closed = mne.io.read_raw_edf(paths[1], preload=True)
for raw in (eyes_open, eyes_closed):
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    raw.set_eeg_reference('average', projection=False)
    raw.filter(1, 30)

先查看少量原始波形。真实 EEG 中的振幅突变、漂移和高频活动可能来自眼动、肌电或电极，而不必然是脑活动。

In [ ]:
eyes_open.plot(duration=10, n_channels=20, scalings='auto')

比较枕区功率谱。闭眼记录通常在 8–13 Hz 附近出现更强 Alpha 峰，但个体峰频和强度会变化。

In [ ]:
picks = [ch for ch in ['O1', 'Oz', 'O2', 'PO3', 'PO4'] if ch in eyes_open.ch_names]
open_psd = eyes_open.compute_psd(fmin=1, fmax=30, picks=picks)
closed_psd = eyes_closed.compute_psd(fmin=1, fmax=30, picks=picks)
fig, ax = plt.subplots(figsize=(8, 4))
open_psd.plot(axes=ax, show=False, color='#4c78a8', spatial_colors=False)
closed_psd.plot(axes=ax, show=False, color='#f58518', spatial_colors=False)
ax.set_title('Posterior PSD: eyes open (blue) vs eyes closed (orange)')
plt.show()

## 思考题

1. 换一名受试者后 Alpha 峰还在 10 Hz 吗？
2. 只看全通道平均会不会掩盖枕区效应？
3. 重参考方式改变后，绝对功率和空间分布怎样变化？

> 这个例子演示的是群体中常见的生理现象，不用于判断个人健康状态。